# 🇱🇰 Piper Sinhala TTS — Fine-tune from Hindi

Fine-tunes the Hindi (rohan) medium Piper checkpoint on a better Sinhala dataset.

**Strategy:** Hindi & Sinhala share similar phoneme structures (both Indic). Starting from a trained Hindi voice gives us prosody, rhythm, and speech quality — we just adapt it to Sinhala phonemes.

**Target:** 100 epochs (~1 hour on L4)

## 1. SSH Tunnel (for remote monitoring)

In [ ]:
# Setup SSH access
!apt-get update -qq && apt-get install -y -qq openssh-server > /dev/null 2>&1
!mkdir -p /root/.ssh
!echo "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIBh+tpMJWdJi42B3aqFJez8UITWK0mCHMsjH5LMJESnl chanclaw@ubuntu-4gb-hel1-2" > /root/.ssh/authorized_keys
!chmod 700 /root/.ssh && chmod 600 /root/.ssh/authorized_keys
!echo "PermitRootLogin yes" >> /etc/ssh/sshd_config
!service ssh start

# Start tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

import subprocess, re, time
proc = subprocess.Popen(['/tmp/cloudflared', 'tunnel', '--url', 'ssh://localhost:22'],
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
time.sleep(8)
for _ in range(20):
    line = proc.stderr.readline()
    m = re.search(r'https://([^\s]+\.trycloudflare\.com)', line)
    if m:
        host = m.group(1)
        print(f"\n🔗 Tunnel: {host}\n")
        break

In [ ]:
# Keep-alive (run this to prevent idle disconnect)
import IPython
IPython.display.display(IPython.display.Javascript('''
setInterval(() => { document.querySelector("colab-connect-button")?.click(); }, 60000);
console.log("Keep-alive started");
'''))

## 2. Install Dependencies

In [ ]:
%%time
# Miniconda + Python 3.10
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/conda
import os
os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda create -y -n piper python=3.10 -q 2>&1 | tail -1
print("✅ Conda ready")

In [ ]:
%%time
%%bash
eval "$(/opt/conda/bin/conda shell.bash hook)" && conda activate piper
set -e

# piper-phonemize from source
cd /tmp
wget -q https://github.com/rhasspy/piper-phonemize/releases/download/2023.11.14-4/piper-phonemize_linux_x86_64.tar.gz
mkdir -p /tmp/piper_phonemize
tar xzf piper-phonemize_linux_x86_64.tar.gz -C /tmp/piper_phonemize
mv /tmp/piper_phonemize/piper_phonemize/* /tmp/piper_phonemize/ 2>/dev/null || true
export LD_LIBRARY_PATH="/tmp/piper_phonemize/lib:$LD_LIBRARY_PATH"

wget -q https://github.com/microsoft/onnxruntime/releases/download/v1.14.1/onnxruntime-linux-x64-1.14.1.tgz
tar xzf onnxruntime-linux-x64-1.14.1.tgz
git clone -q --depth 1 https://github.com/rhasspy/espeak-ng.git /tmp/espeak-ng-src
git clone -q --depth 1 https://github.com/rhasspy/piper-phonemize.git /tmp/piper-phonemize-src
cd /tmp/piper-phonemize-src
pip install -q pybind11 setuptools wheel
CXXFLAGS="-I/tmp/piper_phonemize/include -I/tmp/onnxruntime-linux-x64-1.14.1/include -I/tmp/espeak-ng-src/src/include" \
LDFLAGS="-L/tmp/piper_phonemize/lib" python setup.py bdist_wheel 2>&1 | tail -1
pip install dist/*.whl
python -c "from piper_phonemize import phonemize_espeak; print('✅ piper-phonemize')"

In [ ]:
%%time
%%bash
eval "$(/opt/conda/bin/conda shell.bash hook)" && conda activate piper
set -e

# PyTorch + training deps
pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
pip install -q librosa cython onnxruntime onnx
pip install -q pip==24.0
pip install -q "torchmetrics<0.12"
pip install -q pytorch-lightning==1.7.7 2>&1 | tail -1

# Piper training code
git clone -q https://github.com/rhasspy/piper.git /content/piper
cd /content/piper/src/python && pip install --no-deps -e . 2>&1 | tail -1
cd piper_train/vits/monotonic_align && mkdir -p monotonic_align && cython core.pyx
gcc -shared -pthread -fPIC -fwrapv -O2 \
  $(python -c "import sysconfig; print(sysconfig.get_config_var('CFLAGS'))") \
  -I$(python -c "import numpy; print(numpy.get_include())") \
  -I$(python -c "import sysconfig; print(sysconfig.get_path('include'))") \
  -o monotonic_align/core$(python -c "import sysconfig; print(sysconfig.get_config_var('EXT_SUFFIX'))") core.c

# PL patches
PL="/opt/conda/envs/piper/lib/python3.10/site-packages/pytorch_lightning"
find "$PL" -name "*.py" -exec grep -l "np\.Inf" {} \; | xargs -r sed -i 's/np\.Inf/np.inf/g'
python3 -c "
with open('$PL/core/optimizer.py') as f: c=f.read()
c=c.replace('def _validate_scheduler_api(lr_scheduler_configs','def _validate_scheduler_api(lr_scheduler_configs, model=None):\n    return\ndef _validate_scheduler_api_OLD(lr_scheduler_configs')
with open('$PL/core/optimizer.py','w') as f: f.write(c)
print('✅ All deps installed + patched')
"
python -c "from piper_train.vits.models import SynthesizerTrn; print('✅ VitsModel OK')"

## 3. Download Pretrained Hindi Checkpoint

In [ ]:
%%bash
set -e
pip install -q huggingface_hub

mkdir -p /content/pretrained
cd /content/pretrained

# Download Hindi rohan medium checkpoint
python3 -c "
from huggingface_hub import hf_hub_download, list_repo_tree
import os

# List files in the Hindi checkpoint dir
files = list(list_repo_tree('rhasspy/piper-checkpoints', path_in_repo='hi/hi_IN/rohan/medium', repo_type='dataset'))
print('Files found:')
for f in files:
    print(f'  {f.rfilename if hasattr(f, \"rfilename\") else f}')

# Download checkpoint and config
for f in files:
    name = f.rfilename if hasattr(f, 'rfilename') else str(f)
    if name.endswith('.ckpt') or name.endswith('.json'):
        print(f'Downloading {name}...')
        hf_hub_download('rhasspy/piper-checkpoints', filename=name, repo_type='dataset', local_dir='/content/pretrained')
"
echo "✅ Hindi checkpoint downloaded"
find /content/pretrained -name "*.ckpt" -o -name "*.json" | head -5
du -sh /content/pretrained/

## 4. Download & Prepare Sinhala Dataset

In [ ]:
%%time
%%bash
eval "$(/opt/conda/bin/conda shell.bash hook)" && conda activate piper
pip install -q datasets soundfile

python3 << 'PYDATA'
from datasets import load_dataset
import soundfile as sf
import os

print("Loading eshangj/SinhalaTTS...")
ds = load_dataset("eshangj/SinhalaTTS", split="train")
print(f"Total samples: {len(ds)}")
print(f"Columns: {ds.column_names}")
print(f"First sample: {ds[0]}")

# Prepare in LJSpeech format
os.makedirs("/content/sinhala_dataset/wav", exist_ok=True)
metadata = []
count = 0

for i, sample in enumerate(ds):
    # Adapt based on column names
    text = sample.get("text") or sample.get("sentence") or sample.get("transcription", "")
    audio = sample.get("audio")
    
    if not text or not audio:
        continue
    
    wav_path = f"/content/sinhala_dataset/wav/{i:05d}.wav"
    
    # Save audio as 22050Hz mono
    import numpy as np
    audio_array = np.array(audio["array"], dtype=np.float32)
    sr = audio["sampling_rate"]
    
    # Resample if needed
    if sr != 22050:
        import librosa
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=22050)
    
    sf.write(wav_path, audio_array, 22050)
    metadata.append(f"{i:05d}|{text.strip()}")
    count += 1
    
    if count % 500 == 0:
        print(f"  Processed {count}...")

with open("/content/sinhala_dataset/metadata.csv", "w") as f:
    f.write("\n".join(metadata) + "\n")

print(f"\n✅ Dataset ready: {count} utterances")
print(f"Sample: {metadata[0]}")
PYDATA

## 5. Preprocess for Piper

In [ ]:
%%time
%%bash
eval "$(/opt/conda/bin/conda shell.bash hook)" && conda activate piper
export LD_LIBRARY_PATH="/tmp/piper_phonemize/lib:$LD_LIBRARY_PATH"
export ESPEAK_DATA_PATH="/tmp/piper_phonemize/share/espeak-ng-data"

mkdir -p /content/training_output

python -m piper_train.preprocess \
  --language si \
  --input-dir /content/sinhala_dataset \
  --output-dir /content/training_output \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050 2>&1 | tail -5

echo "✅ Preprocessed: $(wc -l < /content/training_output/dataset.jsonl) entries"

## 6. Fine-tune from Hindi Checkpoint

In [ ]:
%%bash
eval "$(/opt/conda/bin/conda shell.bash hook)" && conda activate piper
export LD_LIBRARY_PATH="/tmp/piper_phonemize/lib:$LD_LIBRARY_PATH"
export ESPEAK_DATA_PATH="/tmp/piper_phonemize/share/espeak-ng-data"

# Find the Hindi checkpoint
HINDI_CKPT=$(find /content/pretrained -name "*.ckpt" | head -1)
echo "Fine-tuning from: $HINDI_CKPT"
echo "Dataset: $(wc -l < /content/training_output/dataset.jsonl) entries"

# Setup rclone for checkpoint backup
apt-get install -y -qq rclone cron > /dev/null 2>&1
mkdir -p /root/.config/rclone
cat > /root/.config/rclone/rclone.conf << 'RCLONECONF'
[drive]
type = drive
scope = drive.file
token = {"access_token":"PASTE_ACCESS_TOKEN","token_type":"Bearer","refresh_token":"PASTE_REFRESH_TOKEN","expiry":"2026-03-18T06:53:54.827645Z"}
RCLONECONF

# Sync script
mkdir -p /content/scripts /content/checkpoint_staging
cat > /content/scripts/rclone_sync.sh << 'SYNCCRON'
#!/bin/bash
STAGING="/content/checkpoint_staging"
mkdir -p "$STAGING"
LATEST=$(find /content/training_output/lightning_logs -name "*.ckpt" -printf '%T@ %p\n' 2>/dev/null | sort -n | tail -1 | awk '{print $2}')
if [ -n "$LATEST" ]; then
  BASENAME=$(basename "$LATEST")
  cp "$LATEST" "$STAGING/$BASENAME" 2>/dev/null
  if [ -f "$STAGING/$BASENAME" ]; then
    rclone copy "$STAGING/" drive:colab-checkpoints/sinhala-finetune/ --ignore-checksum --transfers 4 2>/dev/null
    rm -f "$STAGING"/*.ckpt
    echo "$(date): synced $BASENAME" >> /content/rclone_sync.log
  fi
fi
SYNCCRON
chmod +x /content/scripts/rclone_sync.sh
echo "*/10 * * * * /content/scripts/rclone_sync.sh" | crontab -
service cron start

echo "🚀 Starting fine-tuning (100 epochs)..."

python -m piper_train \
  --dataset-dir /content/training_output \
  --accelerator gpu --devices 1 \
  --batch-size 32 --validation-split 0.05 \
  --num-test-examples 5 --max_epochs 100 \
  --checkpoint-epochs 10 --precision 32 \
  --quality medium \
  --resume_from_checkpoint "$HINDI_CKPT" \
  2>&1 | tail -20

echo "✅ Training complete!"

## 7. Export to ONNX

In [ ]:
%%bash
eval "$(/opt/conda/bin/conda shell.bash hook)" && conda activate piper
export LD_LIBRARY_PATH="/tmp/piper_phonemize/lib:$LD_LIBRARY_PATH"
export ESPEAK_DATA_PATH="/tmp/piper_phonemize/share/espeak-ng-data"

CKPT=$(find /content/training_output/lightning_logs -name "*.ckpt" -printf '%T@ %p\n' | sort -n | tail -1 | awk '{print $2}')
echo "Exporting: $CKPT"

mkdir -p /content/piper_model
python -m piper_train.export_onnx "$CKPT" /content/piper_model/si_LK-sinhala-medium.onnx

# Create config
python3 -c "
import json
config = {
    'audio': {'sample_rate': 22050},
    'espeak': {'voice': 'si'},
    'inference': {'noise_scale': 0.667, 'length_scale': 1.0, 'noise_w': 0.8},
    'language': {'code': 'si', 'family': 'si', 'region': 'LK', 'name_native': 'සිංහල', 'name_english': 'Sinhala', 'country_english': 'Sri Lanka'},
    'model': 'si_LK-sinhala-medium',
    'num_speakers': 1,
    'phoneme_type': 'espeak',
    'phoneme_map': {}
}
with open('/content/piper_model/si_LK-sinhala-medium.onnx.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print('✅ Config created')
"

ls -lh /content/piper_model/

# Upload to Drive
rclone copy /content/piper_model/ drive:colab-checkpoints/sinhala-finetune/model/ --ignore-checksum
echo "✅ Model uploaded to Drive: colab-checkpoints/sinhala-finetune/model/"

## 8. Test Synthesis

In [ ]:
%%bash
eval "$(/opt/conda/bin/conda shell.bash hook)" && conda activate piper
export LD_LIBRARY_PATH="/tmp/piper_phonemize/lib:$LD_LIBRARY_PATH"
export ESPEAK_DATA_PATH="/tmp/piper_phonemize/share/espeak-ng-data"

mkdir -p /content/test_output

python3 << 'PYTEST'
import numpy as np
import onnxruntime as ort
import json, wave
from piper_phonemize import phonemize_espeak

session = ort.InferenceSession("/content/piper_model/si_LK-sinhala-medium.onnx")
with open("/content/piper_model/si_LK-sinhala-medium.onnx.json") as f:
    config = json.load(f)

# Load phoneme_id_map from training config
with open("/content/training_output/config.json") as f:
    train_config = json.load(f)
id_map = train_config["phoneme_id_map"]

sentences = [
    "සිංහල භාෂාවෙන් කතා කරමු",
    "ආයුබෝවන් ශ්‍රී ලංකාව",
    "මේ පරිගණක තාක්ෂණය ඉතා සුන්දරයි",
    "අද කාලගුණය ඉතා හොඳයි",
    "මම ඔබට උදව් කරන්නම්",
]

for i, text in enumerate(sentences):
    phonemes = phonemize_espeak(text, "si")
    phoneme_str = list(phonemes[0])
    
    phoneme_ids = [id_map["^"][0]]
    for p in phoneme_str:
        if p in id_map:
            phoneme_ids.extend(id_map[p])
            phoneme_ids.extend(id_map["_"])
    phoneme_ids.append(id_map["$"][0])
    
    input_array = np.array([phoneme_ids], dtype=np.int64)
    input_lengths = np.array([len(phoneme_ids)], dtype=np.int64)
    scales = np.array([0.667, 1.0, 0.8], dtype=np.float32)
    
    audio = session.run(None, {"input": input_array, "input_lengths": input_lengths, "scales": scales})[0]
    audio_int16 = (audio.squeeze() * 32767).astype(np.int16)
    
    path = f"/content/test_output/finetune_test_{i+1}.wav"
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(22050)
        wf.writeframes(audio_int16.tobytes())
    
    duration = len(audio_int16) / 22050
    print(f"✅ {i+1}. {text} → {duration:.1f}s")

print("\n🎉 All tests done!")
PYTEST

In [ ]:
# Play the test audio
from IPython.display import Audio, display
import glob

for wav in sorted(glob.glob("/content/test_output/finetune_test_*.wav")):
    print(wav.split("/")[-1])
    display(Audio(wav))